In [1]:
import pandas as pd
import glob
import os
import numpy as np

In [2]:
# visualizar todas las columnas 
pd.set_option('display.max_columns', None)

In [3]:
anio_term = "2023_2S"

In [4]:
list_files = glob.glob(os.path.join("..", "data", "desercion", anio_term, "*.csv"))
list_files

['..\\data\\desercion\\2023_2S\\COLEGIO_202601071509.csv',
 '..\\data\\desercion\\2023_2S\\FAM_EST_202601071631.csv',
 '..\\data\\desercion\\2023_2S\\NEXT_REGISTROS_202601091451.csv',
 '..\\data\\desercion\\2023_2S\\_SELECT_ha_anio_ha_termino_ha_cod_estudiante_ha_COD_MATERIA_ACAD_202601091607.csv',
 '..\\data\\desercion\\2023_2S\\_with_calificaciones_as_select_distinct_p_numeroidentificacion_m_202601071640.csv',
 '..\\data\\desercion\\2023_2S\\_WITH_periodo_interes_AS_SELECT_ha_anio_ha_termino_ha_cod_estudi_202601091531.csv']

In [5]:
list_files[0]

'..\\data\\desercion\\2023_2S\\COLEGIO_202601071509.csv'

## Carga de datos

In [6]:
df_dificulad_materia = pd.read_csv("../data/desercion/dificultad_materias.csv")
df_dificulad_materia['CODIGOMATERIA'] = df_dificulad_materia['CODIGOMATERIA'].str.strip()

In [7]:
df_socioeconomico = pd.read_csv("../data/desercion/socioeconomico_17938.csv")

C:\Users\saraujo\AppData\Local\Temp\ipykernel_88264\2157926340.py:1: DtypeWarning: Columns (10,41,42,66,123) have mixed types. Specify dtype option on import or set low_memory=False.
  df_socioeconomico = pd.read_csv("../data/desercion/socioeconomico_17938.csv")


In [8]:
for file in list_files:
    if "_WITH_periodo" in file:
        print("desertores en el periodo", anio_term)
        df_promedio_historico = pd.read_csv(file)
        print(df_promedio_historico.shape)
    elif "_SELECT_ha_anio" in file:
        print("Materias tomadas en el anio", anio_term)
        df_materias_tomadas = pd.read_csv(file)
        df_materias_tomadas['COD_MATERIA_ACAD'] = df_materias_tomadas['COD_MATERIA_ACAD'].str.strip()
        print(df_materias_tomadas.shape)
    elif "NEXT_" in file:
        print("promedio historico academico siguiente al anio", anio_term)
        df_desertores = pd.read_csv(file)
        print(df_desertores.shape)
        # VECES es el numero de veces registradas en los proximos 4 semestres
    elif "COLEGIO_" in file:
        print("datos colegio", anio_term)
        df_colegio = pd.read_csv(file)
        # Hacer un distinct =======================================
        print(df_colegio.shape)
    elif "FAM_EST_" in file:
        print("datos familiares y estuctura", anio_term)
        df_fam_est = pd.read_csv(file)
        # Hacer un distinct =======================================
        print(df_fam_est.shape)
    elif "_with_calificaciones":
        print("calificaicones estudiantes desde el pregrado", anio_term)
        df_calificaciones_pre = pd.read_csv(file)
        print(df_calificaciones_pre.shape)

C:\Users\saraujo\AppData\Local\Temp\ipykernel_88264\1820687994.py:23: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  df_fam_est = pd.read_csv(file)


datos colegio 2023_2S
(8050, 6)
datos familiares y estuctura 2023_2S
(34638, 17)
promedio historico academico siguiente al anio 2023_2S
(1055, 3)
Materias tomadas en el anio 2023_2S
(32979, 8)
calificaicones estudiantes desde el pregrado 2023_2S
(919, 2)
desertores en el periodo 2023_2S
(7997, 8)


#### Promedio histórico por estudiante

In [9]:
df_promedio_historico.dtypes

COD_ESTUDIANTE        int64
PROMEDIO_GENERAL    float64
MAT_TOMADAS           int64
PROMEDIO_AP         float64
ESNOVATO             object
MAT_TOMADAS_2V      float64
MAT_TOMADAS_3V      float64
MAT_REPROBADAS      float64
dtype: object

In [10]:
df_promedio_historico["COD_ESTUDIANTE"] = df_promedio_historico["COD_ESTUDIANTE"].astype(str)
df_promedio_historico["PROMEDIO_GENERAL"] = df_promedio_historico["PROMEDIO_GENERAL"].round(2)
df_promedio_historico["PROMEDIO_AP"] = df_promedio_historico["PROMEDIO_AP"].round(2)


In [11]:
# df_promedio_historico[df_promedio_historico["COD_ESTUDIANTE"] == "201614252"]

In [12]:
df_promedio_historico

,COD_ESTUDIANTE,PROMEDIO_GENERAL,MAT_TOMADAS,PROMEDIO_AP,ESNOVATO,MAT_TOMADAS_2V,MAT_TOMADAS_3V,MAT_REPROBADAS
0,202300216,0.00,0,0.00,S,0.0,0.0,0.0
1,202305736,0.00,0,0.00,S,0.0,0.0,0.0
2,202305983,0.00,0,0.00,S,0.0,0.0,0.0
3,202305991,0.00,0,0.00,S,0.0,0.0,0.0
4,202306155,0.00,0,0.00,S,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...
7992,201715067,8.26,56,8.26,N,NaN,NaN,NaN
7993,201808060,7.36,57,7.36,N,NaN,NaN,NaN
7994,201713666,7.53,57,7.53,N,NaN,NaN,NaN
7995,201609518,7.15,63,7.15,N,NaN,NaN,NaN


In [13]:
df_promedio_historico["ESNOVATO"].value_counts()

ESNOVATO
N    7098
S     899
Name: count, dtype: int64

In [14]:
df_promedio_historico["COD_ESTUDIANTE"].nunique(), df_promedio_historico.shape[0]

(7997, 7997)

In [15]:
# Estudiantes repetidos
df_promedio_historico["COD_ESTUDIANTE"].value_counts()[df_promedio_historico["COD_ESTUDIANTE"].value_counts() > 1].shape

(0,)

#### Carga de calificaciones pregrado

In [16]:
df_calificaciones_pre.shape

(919, 2)

In [17]:
df_calificaciones_pre.keys()

Index(['CODESTUDIANTE', 'PROMEDIOGENERAL'], dtype='object')

In [18]:
df_calificaciones_pre["PROMEDIOGENERAL"] = df_calificaciones_pre["PROMEDIOGENERAL"].round(2)
df_calificaciones_pre["CODESTUDIANTE"] = df_calificaciones_pre["CODESTUDIANTE"].astype(str)

In [19]:
# validar que coincidan los estudiantes de pre
print("Cantidad de estudiantes novatos desde el pre:", df_promedio_historico[df_promedio_historico["ESNOVATO"] == "S"].shape[0], "Cantidad de calificaciones pregrado desde dataframe historico:", df_calificaciones_pre.shape[0])
print("conididencia?", df_promedio_historico[df_promedio_historico["ESNOVATO"] == "S"].shape[0] == df_calificaciones_pre.shape[0])

if df_promedio_historico[df_promedio_historico["ESNOVATO"] == "S"].shape[0] == df_calificaciones_pre.shape[0]:
    print("Los estudiantes coinciden?")
    df_merged = pd.merge(df_promedio_historico[df_promedio_historico["ESNOVATO"] == "S"], df_calificaciones_pre, left_on="COD_ESTUDIANTE", right_on="CODESTUDIANTE", how="inner")
    print("Cantidad de estudiantes novatos desde el pre en el merge:", df_merged.shape[0])

Cantidad de estudiantes novatos desde el pre: 899 Cantidad de calificaciones pregrado desde dataframe historico: 919
conididencia? False


In [20]:
# Crear un mapeo (Serie) a partir de df_calificaciones_pre
mapping_promedio = df_calificaciones_pre.set_index('CODESTUDIANTE')['PROMEDIOGENERAL']

# Identificar las filas que cumplen la condición de S ser novatos
mask_novatos = df_promedio_historico["ESNOVATO"] == "S"

# Actualizar PROMEDIO_GENERAL solo en esas filas, mapeando por COD_ESTUDIANTE
# Se usa 'fillna' al final por si algún estudiante no tiene coincidencia en el df_calificaciones_pre,
# para mantener su valor original.
df_promedio_historico.loc[mask_novatos, "PROMEDIO_GENERAL"] = df_promedio_historico.loc[mask_novatos, "COD_ESTUDIANTE"].map(mapping_promedio).fillna(df_promedio_historico.loc[mask_novatos, "PROMEDIO_GENERAL"])

In [21]:
# Cantidad de promedios nulos que debe ser 0
df_promedio_historico[df_promedio_historico["ESNOVATO"] == "S"][df_promedio_historico[df_promedio_historico["ESNOVATO"] == "S"]["PROMEDIO_GENERAL"].isna()].shape[0]

0

#### Materias tomadas por estudiante

In [22]:
df_materias_tomadas.head()

,ANIO,TERMINO,COD_ESTUDIANTE,COD_MATERIA_ACAD,VEZ_TOMADA,COMPLEMENTARIA,MATERIA_INTEGRADORA,IDPERIODO
0,2023,2S,202001186,ACUG1039,1,S,N,590
1,2023,2S,201802832,ACUG1039,1,S,N,590
2,2023,2S,201401548,ACUG1039,1,S,N,590
3,2023,2S,202111050,ALIG1044,1,S,N,590
4,2023,2S,201902202,ALIG1044,1,S,N,590


In [23]:
df_materias_tomadas["COD_ESTUDIANTE"] = df_materias_tomadas["COD_ESTUDIANTE"].astype(str)
df_materias_tomadas["TERMINO"] = df_materias_tomadas["TERMINO"].astype(str).str.strip()
df_materias_tomadas["MATERIA_INTEGRADORA"] = df_materias_tomadas["MATERIA_INTEGRADORA"].astype(str).str.strip()


In [24]:
df_materias_tomadas["COD_ESTUDIANTE"].nunique(), df_materias_tomadas.shape[0]

(8034, 32979)

In [25]:
df_materias_tomadas["MATERIA_INTEGRADORA"].value_counts()

MATERIA_INTEGRADORA
N     32253
AP      693
RP       33
Name: count, dtype: int64

In [26]:
# MATERIA_INTEGRADORA en las materias tomadas
print("Estudiantes que estan tomado MATERIA_INTEGRADORA:",df_materias_tomadas["MATERIA_INTEGRADORA"].value_counts().sum() - int(df_materias_tomadas["MATERIA_INTEGRADORA"].value_counts()["N"]))

Estudiantes que estan tomado MATERIA_INTEGRADORA: 726


Codigo de las materias integradoras:

In [27]:
df_materias_tomadas[df_materias_tomadas["MATERIA_INTEGRADORA"] != 'N']["COD_MATERIA_ACAD"].unique()

array(['DING2032', 'DIGG2034', 'CCPG1035', 'ELEG1042', 'TELG1033',
       'TLMG1031', 'EYAG1039', 'BIOG1027', 'AGRG1029', 'NUTG2031',
       'ALIG1040', 'MTRG1034', 'INDG1043', 'GEOG1039', 'CIVG1054',
       'NAVG1034', 'OCEG1043', 'ACUG1058', 'CADG2043', 'LOGG1027',
       'ADMG2031', 'ECOG2053', 'AUDG2040', 'TURG2029', 'ARQG2024',
       'QUIG1050', 'MECG1066', 'MCTG1028', 'ESTG1062', 'MING1039',
       'MATG1086'], dtype=object)

#### Estudiantes que desertaron

In [28]:
df_desertores["COD_ESTUDIANTE"] = df_desertores["COD_ESTUDIANTE"].astype(str)

In [29]:
df_desertores["COD_ESTUDIANTE"].nunique(), df_desertores.shape[0]

(1055, 1055)

In [30]:
df_desertores.head(7)

,COD_ESTUDIANTE,INTEGRADORA,VECES
0,,N,0
1,199903519,N,0
2,200110286,N,0
3,200124667,N,0
4,200424356,N,0
5,200500361,N,0
6,200521581,N,0


In [31]:
df_desertores["VECES"].value_counts()

VECES
0    1055
Name: count, dtype: int64

In [32]:
df_desertores["INTEGRADORA"].value_counts()

INTEGRADORA
N    1055
Name: count, dtype: int64

In [33]:
"Desertores", df_desertores["COD_ESTUDIANTE"].nunique(), "Vieron materia integradora", df_materias_tomadas[df_materias_tomadas["MATERIA_INTEGRADORA"] != "N"].shape[0], "AP", df_materias_tomadas[df_materias_tomadas["MATERIA_INTEGRADORA"] == "AP"].shape[0], "RP", df_materias_tomadas[df_materias_tomadas["MATERIA_INTEGRADORA"] == "RP"].shape[0]

('Desertores', 1055, 'Vieron materia integradora', 726, 'AP', 693, 'RP', 33)

Descartando de deserción aquellos que vieron materia integradora y la aprobaron

In [34]:
df_desertores = df_desertores[~df_desertores["COD_ESTUDIANTE"].isin(df_materias_tomadas[df_materias_tomadas["MATERIA_INTEGRADORA"] == "AP"]["COD_ESTUDIANTE"].unique())].copy()
df_desertores.shape

(368, 3)

In [35]:
print("Al menos los RP", df_materias_tomadas[df_materias_tomadas["MATERIA_INTEGRADORA"] == "RP"].shape[0], "deberian tener integradora en desercion:", df_desertores[df_desertores["INTEGRADORA"] == "S"].shape[0], "y si no lo tienen son desertores")

Al menos los RP 33 deberian tener integradora en desercion: 0 y si no lo tienen son desertores


In [36]:
cant_est_rp_desertores = df_desertores[df_desertores["COD_ESTUDIANTE"].isin(df_materias_tomadas[df_materias_tomadas["MATERIA_INTEGRADORA"] == "RP"]["COD_ESTUDIANTE"].unique())].shape[0]
print("Estudiantes que reprobaron la materia integradora y desertaron:", cant_est_rp_desertores, ", los que no estan puede que si hayna vuelto a ver la materia")

Estudiantes que reprobaron la materia integradora y desertaron: 6 , los que no estan puede que si hayna vuelto a ver la materia


In [37]:
ests_rp_no_in_desertores = df_desertores[~df_desertores["COD_ESTUDIANTE"].isin(df_materias_tomadas[df_materias_tomadas["MATERIA_INTEGRADORA"] == "RP"]["COD_ESTUDIANTE"].unique())]
ests_rp_no_in_desertores.shape[0]

362

#### Dificultad de materias

In [38]:
# df_dificulad_materia['DIFICULTAD'] = (df_dificulad_materia['DIFICULTAD'] .str.replace(',', '.', regex=False) .astype(float))

In [39]:
df_dificulad_materia.head()

,CODIGOMATERIA,MATERIA,DIFICULTAD
0,ACUG1035,ACUICULTURA ORNAMENTAL,8.52
1,ACUG1036,ANÁLISIS DE DATOS ACUÍCOLAS,7.72
2,ACUG1037,BIENESTAR ANIMAL,8.38
3,ACUG1039,CULTIVO DE ESPECIES NO TRADICIONALES,8.63
4,ACUG1040,CULTIVO DE PLANCTON,8.18


In [40]:
# ninguna materia se repite?
df_count_materia = df_dificulad_materia["CODIGOMATERIA"].value_counts()
df_count_materia[df_count_materia > 1].shape[0]

0

In [41]:
# df_dificulad_materia.groupby(["ANIO", "TERMINO"]).count()

#### Socioeconómico

In [42]:
df_socioeconomico["CODESTUDIANTE"].dtypes

dtype('int64')

In [43]:
df_socioeconomico["CODESTUDIANTE"] = df_socioeconomico["CODESTUDIANTE"].astype(str)

In [44]:
df_socioeconomico.shape

(17938, 131)

In [45]:
df_socioeconomico.head()

,ESTACOMPLETA,IDPERSONA,FECHACREACION,FECHAENVIOUBEP,FECHAENVIO,CODESTUDIANTE,APELLIDOS,NOMBRES,EMAIL,P1_NACIONALIDAD,NUMEROIDENTIFICACION,ANIO_TERMINO_INGRESO,CATEGORIA,ISE,TIENEDISCAPACIDAD,TIPODISCAPACIDAD,PORCENTAJEDISCAPACIDAD,SEXO,AUTOIDENTIFICACIONGENERO,AUTOIDENTIFICACIONETNICA,ESTADOCIVIL,FECHANACIMIENTO,PAISNACIMIENTO,PROVINCIANACIMIENTO,CIUDADNACIMIENTO,TELEFONOCELULAR,TELEFONOFIJO,CORREOALTERNO,COLEGIO,PAISCOLEGIO,PROVINCIACOLEGIO,CANTONCOLEGIO,TIPOCOLEGIO,ANIOGRADUACION,CATEGORIACOLEGIO,JORNADA,BECACOLEGIO,OTROSIDIOMAS,IDIOMAS,COBOCIMIENTOINGLESPOR,COMIDASALDIA,FRASETRESCOMIDAS,FRASEHABITOALIMENTICIO,FRASEHABITOALIMENTICIOV2,CONSUMODIARIOESPOL,TIEMPOPROMEDIOLLEGARESPOL,VECESBUSENTRADA,VECESCARROENTRADA,BICICLETAENTRADA,TIEMPOPROMEDIOBICICLETAENTRADAESPOL,VECESTAXIENTRADA,VECESCARROMOTOAMIGOENTRADA,CAMINAENTRADA,TIEMPOPROMEDIOCAMINATAENTRADAESPOL,VECESTRICIMOTOENTRADA,VECESBUSSALIDA,VECESCARROSALIDA,BICICLETASALIDA,TIEMPOPROMEDIOBICICLETASALIDAESPOL,VECESTAXISALIDA,VECESCARROMOTOAMIGOSALIDA,CAMINASALIDA,TIEMPOPROMEDIOCAMINATASALIDAESPOL,VECESTRICIMOTOSALIDA,NIVELINGLES,POSEETARJETACREDITO,POSEETARJETADEBITO,CUENTASBANCO,HERMANOSESTUDIANDOESPOL,ALIMENTACION,TRANSPORTE,SERVICIOS,ARRIENDO,ALICUOTAS,VESTIMENTA,SALUD,EDUCACION,TARJETACREDITO,ENTRETENIMIENTO,OTROS,NIVELINSTRUCCIONPADRE,NIVELINSTRUCCIONMADRE,ESTADOCIVILPADRES,DISCAPACIDAD,FAMILIARDISCAPACIDAD,ENFERMEDAD,FAMILIARENFERMEDAD,RECIBEBONO,DIFICULTADAPRENDIZAJE,PAISVIVE,PROVINCIAVIVE,CIUDADVIVE,PARROQUIAVIVE,DIRECCION,COORDENADAS,TIPOPARROQUIA,VIVEGRUPOFAMILIAR,PAISVIVESEP,PROVINCIAVIVIENDASEP,CANTONVIVESEP,PARROQUIAVIVESEP,DIRECCIONVIVSEP,TIPOVIVIENDASEP,ESTADOVIVIENDASEP,CANTIDADCUARTOS,CANTIDADBANIO,SALA,COMEDOR,ESTUDIO,COCINA,LAVANDERIA,GARAJE,METERIALTECHOVIVIENDASEP,METERIALPISOVIVIENDASEP,METERIALPAREDVIVIENDASEP,VIAACCESOVIVIENDASEP,ABASTECIMIENTOAGUA,SERVHIGIENE,ELIMINACIONBASURA,SERVICIOELECTRICIDAD,POSEEVEHICULO,CANTIDADVEHICULO,RECIBEAYUDA,PARIENTEAYUDA,VALORAYUDA,TIPOBACHILLER,COMIDAS,DISPOSITIVOS,MANEJOCELULAR,ACCESOINTERNET,NUMEROSFAMILIARES
0,1,3057,2023-04-28,NaN,2022-04-22-22.03.56.450305,200004851,ESCOBAR SEGOVIA,KENNY FERNANDO,kescobar@espol.edu.ec,ECUATORIANA,921620548,2000 1S,4.0,0.7801,N,NaN,0.0,Masculino,masculino,Mestizo,casado,1982-06-17,ECUADOR,GUAYAS,GUAYAQUIL,95792616; 95792616,2651491,kescobarsegovia@gmail.com,ACADEMIA NAVAL GUAYAQUIL GUAYAQUIL,ECUADOR,GUAYAS,GUAYAQUIL,Particular,2000.0,0.0,MATUTINA (07:00-14:00 APROX.),Ninguna,SI,INGLÉS,Aprendizaje del colegio,3,No se acostumbra en mi hogar servir comida tre...,Compro en bares de ESPOL,NaN,5,16 a 30 minutos,0,2,0,NaN,0,0,0,NaN,0,0,2,0,NaN,0,0,0,NaN,0,INTERMEDIO,SI,SI,NaN,NO,500,100,250,0,0,50,40,20,500,100,100,Secundaria Completa,Superior Universitaria incompleta,Separados[Unión de Hecho],NO,NaN,SI,Otro:No tiene enfermedad;Madre:No tiene enferm...,NO,NaN,ECUADOR,GUAYAS,GUAYAQUIL,TARQUI,COOP PANCHO JACOME,",",URBANA,SI,NaN,NaN,NaN,NaN,NaN,Casa/Villa,"Propia, totalmente pagada (ha sido totalmente ...",3,2,SI,SI,NO,SI,NO,SI,Zinc /teja /eternit,Cerámica / Baldosa / Vinil,Hormigón / ladrillo / bloque / cemento,Carretera / calle pavimentada o adoquinada / c...,RED PÚBLICA LAS 24 HORAS,CONECTADO A RED DE ALCANTARILLADO,CARRO RECOLECTAOR,EP. CON MEDIDOR EXCLUSIVO DE VIVIENDA,SI,NaN,NO,NaN,0.0,Bachillerato en Ciencias,DESAYUNO;ALMUERZO;MERIENDA,PORTÁTIL EXCLUSIVA,Plan de datos,WIFI EXCLUSIVO DE LA VIVIENDA,4
1,1,4056,2020-10-28,NaN,2020-10-13-18.27.10.390099,200217933,RECALDE GARAY,MARIA TERESA,trecalde@espol.edu.ec,ECUATORIANA,912190238,2002 1S,4.0,0.8047,N,NaN,0.0,Femenino,femenino,Blanco,soltero,1983-11-17,ECUADOR,GUAYAS,GUAYAQUIL,97214060,2001847; 2354217,teresa.recaldeg@gmail.com,CRUZ DEL SUR GUAYAQUIL,ECUADOR,GUAYAS,GUAYAQUIL,Particular,2002.0,0.0,MATUTINA (07:00-14:00 APROX.),Por situación socioeconómica,SI,INGLÉS,Aprendizaje del colegio,3,NaN,Salgo de ESPOL y voy a comer a casa,NaN,0,15 minutos o menos,0,0,0,NaN,2,0,0,NaN,-1,0,0,0,NaN,0,0,0,NaN,-1,NaN,SI,NaN,NaN,NO,800,250,126,0,0,1

In [46]:
estudiantes_mas_socio = df_socioeconomico["CODESTUDIANTE"].value_counts()[df_socioeconomico["CODESTUDIANTE"].value_counts() > 1]
estudiantes_mas_socio.index

Index(['201615093', '201500461', '201508922', '201300423', '201107917',
       '201911781', '201808821', '201517188', '202006193', '201514249',
       '201804507', '201602844', '201805231', '201604642', '201811189',
       '201405838', '201711033', '201900420', '201700622', '201701273',
       '201508962', '201807641', '200722015', '201710555', '201600897',
       '201602950', '201808250', '201804192', '201604030', '202008546',
       '201809852', '201416541', '200608578', '201510105', '201803392',
       '201515581', '201916798', '201803699', '201401538', '201601218',
       '201912466', '201416531', '201411283', '202004495', '201608239',
       '201807336', '201807385', '201515805', '201703287', '201502566',
       '202001459', '201807682', '201807740', '201512497', '201506748',
       '201710415', '201505290', '201612462', '202009841', '201812922',
       '201808235', '201906526', '201010261', '201609930', '201711470',
       '201609989', '201812419', '201507356', '201607439', '2017

In [47]:
df_socioeconomico[df_socioeconomico["CODESTUDIANTE"].isin(estudiantes_mas_socio.index)].sort_values("CODESTUDIANTE").head()

,ESTACOMPLETA,IDPERSONA,FECHACREACION,FECHAENVIOUBEP,FECHAENVIO,CODESTUDIANTE,APELLIDOS,NOMBRES,EMAIL,P1_NACIONALIDAD,NUMEROIDENTIFICACION,ANIO_TERMINO_INGRESO,CATEGORIA,ISE,TIENEDISCAPACIDAD,TIPODISCAPACIDAD,PORCENTAJEDISCAPACIDAD,SEXO,AUTOIDENTIFICACIONGENERO,AUTOIDENTIFICACIONETNICA,ESTADOCIVIL,FECHANACIMIENTO,PAISNACIMIENTO,PROVINCIANACIMIENTO,CIUDADNACIMIENTO,TELEFONOCELULAR,TELEFONOFIJO,CORREOALTERNO,COLEGIO,PAISCOLEGIO,PROVINCIACOLEGIO,CANTONCOLEGIO,TIPOCOLEGIO,ANIOGRADUACION,CATEGORIACOLEGIO,JORNADA,BECACOLEGIO,OTROSIDIOMAS,IDIOMAS,COBOCIMIENTOINGLESPOR,COMIDASALDIA,FRASETRESCOMIDAS,FRASEHABITOALIMENTICIO,FRASEHABITOALIMENTICIOV2,CONSUMODIARIOESPOL,TIEMPOPROMEDIOLLEGARESPOL,VECESBUSENTRADA,VECESCARROENTRADA,BICICLETAENTRADA,TIEMPOPROMEDIOBICICLETAENTRADAESPOL,VECESTAXIENTRADA,VECESCARROMOTOAMIGOENTRADA,CAMINAENTRADA,TIEMPOPROMEDIOCAMINATAENTRADAESPOL,VECESTRICIMOTOENTRADA,VECESBUSSALIDA,VECESCARROSALIDA,BICICLETASALIDA,TIEMPOPROMEDIOBICICLETASALIDAESPOL,VECESTAXISALIDA,VECESCARROMOTOAMIGOSALIDA,CAMINASALIDA,TIEMPOPROMEDIOCAMINATASALIDAESPOL,VECESTRICIMOTOSALIDA,NIVELINGLES,POSEETARJETACREDITO,POSEETARJETADEBITO,CUENTASBANCO,HERMANOSESTUDIANDOESPOL,ALIMENTACION,TRANSPORTE,SERVICIOS,ARRIENDO,ALICUOTAS,VESTIMENTA,SALUD,EDUCACION,TARJETACREDITO,ENTRETENIMIENTO,OTROS,NIVELINSTRUCCIONPADRE,NIVELINSTRUCCIONMADRE,ESTADOCIVILPADRES,DISCAPACIDAD,FAMILIARDISCAPACIDAD,ENFERMEDAD,FAMILIARENFERMEDAD,RECIBEBONO,DIFICULTADAPRENDIZAJE,PAISVIVE,PROVINCIAVIVE,CIUDADVIVE,PARROQUIAVIVE,DIRECCION,COORDENADAS,TIPOPARROQUIA,VIVEGRUPOFAMILIAR,PAISVIVESEP,PROVINCIAVIVIENDASEP,CANTONVIVESEP,PARROQUIAVIVESEP,DIRECCIONVIVSEP,TIPOVIVIENDASEP,ESTADOVIVIENDASEP,CANTIDADCUARTOS,CANTIDADBANIO,SALA,COMEDOR,ESTUDIO,COCINA,LAVANDERIA,GARAJE,METERIALTECHOVIVIENDASEP,METERIALPISOVIVIENDASEP,METERIALPAREDVIVIENDASEP,VIAACCESOVIVIENDASEP,ABASTECIMIENTOAGUA,SERVHIGIENE,ELIMINACIONBASURA,SERVICIOELECTRICIDAD,POSEEVEHICULO,CANTIDADVEHICULO,RECIBEAYUDA,PARIENTEAYUDA,VALORAYUDA,TIPOBACHILLER,COMIDAS,DISPOSITIVOS,MANEJOCELULAR,ACCESOINTERNET,NUMEROSFAMILIARES
127,3,53935,2024-07-17,2021-05-27,2019-05-05-17.30.15.430117,200608578,BAQUE CAMBRIDGE,ANGEL DANIEL,angdabaq@espol.edu.ec,ECUATORIANA,924730229,2006 1S,2.0,0.4698,N,NaN,NaN,Masculino,masculino,Montubio,soltero,1985-07-16,ECUADOR,GUAYAS,GUAYAQUIL,82593416,2805900,angelbaquecambridge@gmail.com,EXPERIMENTAL LEONIDAS GARCIA GUAYAQUIL,ECUADOR,GUAYAS,GUAYAQUIL,Nacional,2003.0,0.0,VESPERTINA (12:00-18:00 APROX.),Ninguna,NO,NaN,NaN,3,NaN,Traigo comida de casa,NaN,0,16 a 30 minutos,1,0,0,NaN,0,0,0,NaN,-1,1,0,0,NaN,0,0,0,NaN,-1,NaN,NO,SI,NaN,NO,218,100,140,0,0,20,20,20,0,0,0,NaN,Secundaria Completa,Viudo/a,NO,NaN,NO,NaN,NO,NaN,ECUADOR,GUAYAS,GUAYAQUIL,CHONGÓN,URBANIZACIÓN BOSQUETTO - CHONGÓN - GUAYAQUIL,NaN,RURAL,SI,NaN,NaN,NaN,NaN,NaN,NaN,Cedida (Si el inmueble es entregado por una pe...,3,2,NO,NO,NO,SI,SI,NO,Zinc /teja /eternit,Cemento / ladrillo,Hormigón / ladrillo / bloque / cemento,Carretera / calle pavimentada o adoquinada / c...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN,NaN,NaN,NaN,NaN,3
128,3,53935,2021-05-29,2021-05-27,2019-05-05-17.30.15.430117,200608578,BAQUE CAMBRIDGE,ANGEL DANIEL,angdabaq@espol.edu.ec,ECUATORIANA,924730229,2006 1S,2.0,0.4570,N,NaN,NaN,Masculino,masculino,Montubio,soltero,1985-07-16,ECUADOR,GUAYAS,GUAYAQUIL,82593416,2805900,angelbaquecambridge@gmail.com,EXPERIMENTAL LEONIDAS GARCIA GUAYAQUIL,ECUADOR,GUAYAS,GUAYAQUIL,Nacional,2003.0,0.0,VESPERTINA (12:00-18:00 APROX.),Ninguna,NO,NaN,NaN,3,NaN,Traigo comida de casa,NaN,0,16 a 30 minutos,1,0,0,NaN,0,0,0,NaN,-1,1,0,0,NaN,0,0,0,NaN,-1,NaN,NO,SI,NaN,NO,218,100,140,0,0,20,20,20,0,0,0,NaN,Secundaria Completa,Viudo/a,NO,NaN,NO,NaN,NO,NaN,ECUADOR,GUAYAS,GUAYAQUIL,CHONGÓN,URBANIZACIÓN BOSQUETTO - CHONGÓN - GUAYAQUIL,NaN,RURAL,SI,NaN,NaN,NaN,NaN,NaN,NaN,Cedida (Si el inmueble es entregado por una pe...,3,2,NO,NO,NO,SI,SI,NO,Zinc /teja /eternit,Cemento / ladrillo,Hormigón / ladrillo / bloque / cemento,Carretera / calle pavimentada o adoquinada / c...,NaN,

In [48]:
df_socioeconomico["ESTACOMPLETA"].shape[0], df_socioeconomico["ESTACOMPLETA"].isna().sum(), df_socioeconomico["ESTACOMPLETA"].shape[0] - df_socioeconomico["ESTACOMPLETA"].isna().sum()

(17938, np.int64(0), np.int64(17938))

In [49]:
# eliminar filas duplicadas en df_socioeconomico y de preferencia ESTACOMPLETA > 0
df_socioeconomico = df_socioeconomico.sort_values("ESTACOMPLETA", ascending=False).drop_duplicates(subset=["CODESTUDIANTE"], keep="first")

In [50]:
print("Cantidad de repetidos:", df_socioeconomico["CODESTUDIANTE"].value_counts()[df_socioeconomico["CODESTUDIANTE"].value_counts() > 1].shape[0])

Cantidad de repetidos: 0


#### Colegio

In [51]:
df_colegio.dtypes

COD_ESTUDIANTE      int64
PAIS               object
PROVINCIA          object
CANTON             object
PENSION           float64
TIPOCOLEGIO        object
dtype: object

In [52]:
# COD_ESTUDIANTE to string
df_colegio["COD_ESTUDIANTE"] = df_colegio["COD_ESTUDIANTE"].astype(str)

In [53]:
df_colegio.head()

,COD_ESTUDIANTE,PAIS,PROVINCIA,CANTON,PENSION,TIPOCOLEGIO
0,201703295,ECUADOR,AZUAY,CUENCA,10.00,Fiscal
1,201914272,ECUADOR,AZUAY,CUENCA,66.80,Particular
2,202107777,ECUADOR,AZUAY,CUENCA,143.79,Fiscomisional
3,202106373,ECUADOR,AZUAY,CUENCA,143.79,Fiscomisional
4,202005005,ECUADOR,AZUAY,CUENCA,83.40,Particular


In [54]:
df_colegio["COD_ESTUDIANTE"].value_counts()[df_colegio["COD_ESTUDIANTE"].value_counts() > 1].shape

(0,)

In [55]:
df_colegio.shape[0]

8050

In [56]:
# eliminar duplicados en caso de haber
df_colegio = df_colegio.sort_values("PENSION", ascending=False).drop_duplicates(subset=["COD_ESTUDIANTE"], keep="first")

In [57]:
df_colegio.shape[0]

8050

#### Familiares y estructura --(falta)

###### PENDIENTE: unir con df_materias_tomadas para tener toda la info en un solo dataframe
CADA FILA ES UN ESTUDIANTE

In [58]:
df_fam_est.head()

,CODESTUDIANTE,COD_ESTUDIANTE,IDPERSONA,PARENTESCO,TIPOIDENTIFICACION,EDAD,NIVELINSTRUCCION,OCUPACION,INGRESOMENSUAL,NIVELAPORTACION,TIPOSEGURO,TIENEDISCAPACIDAD,ENFERMEDADCASTATROFICA,IDENTIFICACION,RUTADISCAPACIDAD,RUTAENFERMEDAD,RUTAENFERMEDADPREEXISTENTE
0,201305056,201305056,645537,Madre,CEDULA,69,Primaria Completa,Desocupados/Desempleado,0,NINGUNO,Ninguno de los anteriores,NaN,NaN,0908332448,no,no,no
1,201305056,201305056,645537,Hermano(a),CEDULA,47,Secundaria Completa,Desocupados/Desempleado,0,NINGUNO,Ninguno de los anteriores,NaN,NaN,0917765653,no,no,no
2,201305056,201305056,645537,Otro,CEDULA,39,Secundaria Completa,"Cuenta propia (Incluye vendedores ambulantes, ...",394,MEDIO,Ninguno de los anteriores,NaN,NaN,0925615304,no,no,no
3,201305056,201305056,645537,Hermano(a),CEDULA,50,Secundaria Completa,"Cuenta propia (Incluye vendedores ambulantes, ...",400,ALTO,Ninguno de los anteriores,NaN,NaN,0916191158,no,no,no
4,201305056,201305056,645537,Yo mismo (estudiante),CEDULA,29,Superior no Universitaria (tecnología o técnic...,ESTUDIAR,425,ALTO,Ninguno de los anteriores,NaN,NaN,0930479589,no,no,no


## Merge

#### Union de las materias tomadas con el colegio (basico)

In [59]:
df_materias_tomadas.shape[0], df_colegio.shape[0]

(32979, 8050)

In [60]:
df_materias_tomadas = pd.merge(df_materias_tomadas, df_colegio, left_on="COD_ESTUDIANTE", right_on="COD_ESTUDIANTE", how="left")
df_materias_tomadas.shape

(32979, 13)

In [61]:
df_materias_tomadas["TIPOCOLEGIO"].isna().sum()

np.int64(3)

In [62]:
# df_materias_tomadas.head(7)

#### Union de las materias tomadas con la dificultad de las materias

In [63]:
df_materias_tomadas.shape[0], df_dificulad_materia.shape[0]

(32979, 927)

In [64]:
result_tomadas_con_dificultad = pd.merge(df_materias_tomadas, df_dificulad_materia, left_on="COD_MATERIA_ACAD", right_on="CODIGOMATERIA", how="left")
result_tomadas_con_dificultad.shape

(32979, 16)

In [65]:
print("Materias sin dificultad:", result_tomadas_con_dificultad[result_tomadas_con_dificultad["DIFICULTAD"].isna()].shape[0])

Materias sin dificultad: 0


In [66]:
result_tomadas_con_dificultad

,ANIO,TERMINO,COD_ESTUDIANTE,COD_MATERIA_ACAD,VEZ_TOMADA,COMPLEMENTARIA,MATERIA_INTEGRADORA,IDPERIODO,PAIS,PROVINCIA,CANTON,PENSION,TIPOCOLEGIO,CODIGOMATERIA,MATERIA,DIFICULTAD
0,2023,2S,202001186,ACUG1039,1,S,N,590,ECUADOR,GUAYAS,GUAYAQUIL,204.12,Particular,ACUG1039,CULTIVO DE ESPECIES NO TRADICIONALES,8.63
1,2023,2S,201802832,ACUG1039,1,S,N,590,ECUADOR,GUAYAS,GUAYAQUIL,0.00,Fiscal,ACUG1039,CULTIVO DE ESPECIES NO TRADICIONALES,8.63
2,2023,2S,201401548,ACUG1039,1,S,N,590,ECUADOR,GUAYAS,GUAYAQUIL,55.00,Particular,ACUG1039,CULTIVO DE ESPECIES NO TRADICIONALES,8.63
3,2023,2S,202111050,ALIG1044,1,S,N,590,ECUADOR,GUAYAS,GUAYAQUIL,101.93,Particular,ALIG1044,PROCESAMIENTO DE FRUTAS Y VEGETALES,8.32
4,2023,2S,201902202,ALIG1044,1,S,N,590,ECUADOR,GUAYAS,GUAYAQUIL,262.90,Particular,ALIG1044,PROCESAMIENTO DE FRUTAS Y VEGETALES,8.32
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32974,2023,2S,202309183,MECT3005,1,S,N,590,ECUADOR,CAÑAR,LA TRONCAL,0.00,Fiscal,MECT3005,ELEMENTOS DE MÁQUINAS,8.43
32975,2023,2S,202309282,MECT3005,1,S,N,590,ECUADOR,SANTA ELENA,SANTA ELENA,20.00,Fiscomisional,MECT3005,ELEMENTOS DE MÁQUINAS,8.43
32976,2023,2S,202309191,MECT3005,1,S,N,590,ECUADOR,GUAYAS,GUAYAQUIL,0.00,Fiscal,MECT3005,ELEMENTOS DE MÁQUINAS,8.43
32977,2023,2S,201704483,MEDP2007,1,N,N,590,ECUADOR,GUAYAS,SAMBORONDON,454.00,Particular,MEDP2007,CONSERVACIÓN DE LA BIODIVERSIDAD,9.46


In [67]:
# Materias que no tienen dificultad asignada, analizar
print("Cantidad de filas que no tienen dificultad:", result_tomadas_con_dificultad["DIFICULTAD"].isna().sum())

Cantidad de filas que no tienen dificultad: 0


In [68]:
list_materias_sin_dificultad = result_tomadas_con_dificultad[result_tomadas_con_dificultad["DIFICULTAD"].isna()]["COD_MATERIA_ACAD"].unique()
print("Cantidad de materias que no tienen dificultad únicas:", len(list_materias_sin_dificultad))

Cantidad de materias que no tienen dificultad únicas: 0


Antes de suavizar o rellenar la dificultad de materias

In [69]:
result_tomadas_con_dificultad[result_tomadas_con_dificultad["COD_MATERIA_ACAD"].isin(list_materias_sin_dificultad)]

,ANIO,TERMINO,COD_ESTUDIANTE,COD_MATERIA_ACAD,VEZ_TOMADA,COMPLEMENTARIA,MATERIA_INTEGRADORA,IDPERIODO,PAIS,PROVINCIA,CANTON,PENSION,TIPOCOLEGIO,CODIGOMATERIA,MATERIA,DIFICULTAD


In [70]:
if len(list_materias_sin_dificultad) > 0:
    print("Codigos de materias sin dificultad:", list_materias_sin_dificultad)
    
    # con integradora la dificultad es la media de las materias que la componen
    # suavizado o llenado de los datos nulos en DIFICULTAD y que sea de integradora (!= N)
    mask_integradora_sin_dificultad = result_tomadas_con_dificultad["DIFICULTAD"].isna() & (result_tomadas_con_dificultad["MATERIA_INTEGRADORA"] != 'N')

    # Calcular la media por ANIO solo de valores NO NULOS
    mean_dificultad_por_anio = result_tomadas_con_dificultad[~result_tomadas_con_dificultad["DIFICULTAD"].isna()].groupby("ANIO")["DIFICULTAD"].mean()
    # # Crear diccionario con media por año (excluyendo nulos)
    # mean_dificultad_por_anio = result_tomadas_con_dificultad.groupby("ANIO")["DIFICULTAD"].mean()

    # Asignar la media a las materias integradoras sin dificultad
    result_tomadas_con_dificultad.loc[mask_integradora_sin_dificultad, "DIFICULTAD"] = result_tomadas_con_dificultad.loc[mask_integradora_sin_dificultad, "ANIO"].map(mean_dificultad_por_anio)
else:
    print("No hay materias sin dificultad para suavizar o rellenar")

No hay materias sin dificultad para suavizar o rellenar


In [71]:
if len(list_materias_sin_dificultad) > 0:
    print("Verificando que tambien tenga MATERIA")
    result_tomadas_con_dificultad.loc[mask_integradora_sin_dificultad, "CODIGOMATERIA"] = result_tomadas_con_dificultad.loc[mask_integradora_sin_dificultad, "COD_MATERIA_ACAD"]
    result_tomadas_con_dificultad.loc[mask_integradora_sin_dificultad, "MATERIA"] = "Integradora agregada"
    print("Y asignando nombre de materia a las integradoras agregadas")

Despues de suavizar o rellenar la dificultad de materias

In [72]:
result_tomadas_con_dificultad[result_tomadas_con_dificultad["COD_MATERIA_ACAD"].isin(list_materias_sin_dificultad)]

,ANIO,TERMINO,COD_ESTUDIANTE,COD_MATERIA_ACAD,VEZ_TOMADA,COMPLEMENTARIA,MATERIA_INTEGRADORA,IDPERIODO,PAIS,PROVINCIA,CANTON,PENSION,TIPOCOLEGIO,CODIGOMATERIA,MATERIA,DIFICULTAD


In [73]:
"Cantidad de materias sin dificultad:", result_tomadas_con_dificultad["DIFICULTAD"].isna().sum()

('Cantidad de materias sin dificultad:', np.int64(0))

In [74]:
result_tomadas_con_dificultad["DIFICULTAD"] = result_tomadas_con_dificultad["DIFICULTAD"].round(2)

#### Una fila por estudiante con el rango de dificultad y vez tomada

In [75]:
# Descartar las que no tienen dificultad asignada, copy() crea una copia independiente del DataFrame
result_tomadas_con_dificultad_sin_nulo = result_tomadas_con_dificultad[~result_tomadas_con_dificultad["DIFICULTAD"].isna()].copy()
result_tomadas_con_dificultad_sin_nulo.shape
# COD_ESTUDIANTE es igual a CODIGOMATERIA

(32979, 16)

In [76]:
result_tomadas_con_dificultad_sin_nulo.dtypes

ANIO                     int64
TERMINO                 object
COD_ESTUDIANTE          object
COD_MATERIA_ACAD        object
VEZ_TOMADA               int64
COMPLEMENTARIA          object
MATERIA_INTEGRADORA     object
IDPERIODO                int64
PAIS                    object
PROVINCIA               object
CANTON                  object
PENSION                float64
TIPOCOLEGIO             object
CODIGOMATERIA           object
MATERIA                 object
DIFICULTAD             float64
dtype: object

In [77]:
# Convert columns to numeric
result_tomadas_con_dificultad_sin_nulo['DIFICULTAD'] = pd.to_numeric(result_tomadas_con_dificultad_sin_nulo['DIFICULTAD'], errors='coerce')
result_tomadas_con_dificultad_sin_nulo['VEZ_TOMADA'] = pd.to_numeric(result_tomadas_con_dificultad_sin_nulo['VEZ_TOMADA'], errors='coerce')

In [78]:
df_estudiantes_resumen = result_tomadas_con_dificultad_sin_nulo.groupby(
    ['COD_ESTUDIANTE', 'ANIO', 'TERMINO', 'IDPERIODO']
).agg(
    CANT_MATERIAS=('COD_MATERIA_ACAD', 'count'),
    CANT_VEZ_1=('VEZ_TOMADA', lambda x: (x == 1).sum()),
    CANT_VEZ_2=('VEZ_TOMADA', lambda x: (x == 2).sum()),
    CANT_VEZ_3=('VEZ_TOMADA', lambda x: (x == 3).sum()),
    
    COMPLEMENTARIA=('COMPLEMENTARIA', 'first'),
    MATERIA_INTEGRADORA=('MATERIA_INTEGRADORA', 'first'),
    PAIS=('PENSION', 'first'),
    PROVINCIA=('PROVINCIA', 'first'),
    CANTON=('CANTON', 'first'),
    PENSION=('PENSION', 'first'),
    TIPOCOLEGIO=('TIPOCOLEGIO', 'first'),
    MATERIA=('MATERIA', 'first'),

    CANT_DIFICULTAD_ALTA=('DIFICULTAD', lambda x: (x <= 6).sum()),
    CANT_DIFICULTAD_MEDIA=('DIFICULTAD', lambda x: ((x > 6) & (x <= 8.5)).sum()),
    CANT_DIFICULTAD_BAJA=('DIFICULTAD', lambda x: (x > 8.5).sum()) # mientras mas alta es mas facil
).reset_index()

df_estudiantes_resumen.shape

(8034, 19)

In [79]:
df_estudiantes_resumen.head(7)

,COD_ESTUDIANTE,ANIO,TERMINO,IDPERIODO,CANT_MATERIAS,CANT_VEZ_1,CANT_VEZ_2,CANT_VEZ_3,COMPLEMENTARIA,MATERIA_INTEGRADORA,PAIS,PROVINCIA,CANTON,PENSION,TIPOCOLEGIO,MATERIA,CANT_DIFICULTAD_ALTA,CANT_DIFICULTAD_MEDIA,CANT_DIFICULTAD_BAJA
0,,2023,2S,590,3,3,0,0,S,N,NaN,None,None,NaN,None,INGLÉS I,0,3,0
1,199201526,2023,2S,590,1,0,0,1,S,N,0.56,GUAYAS,CORONEL MARCELINO MARIDUEÑA,0.56,Particular,TURBOMAQUINARIA Y PLANTAS DE POTENCIA,0,1,0
2,199722414,2023,2S,590,5,5,0,0,S,N,0.16,GUAYAS,GUAYAQUIL,0.16,Fiscal,ANÁLISIS NUTRICIONAL DE LOS ALIMENTOS,0,5,0
3,199903519,2023,2S,590,1,1,0,0,N,AP,1.60,GUAYAS,GUAYAQUIL,1.60,Fiscal,MATERIA INTEGRADORA DE ECONOMÍA,0,1,0
4,200102978,2023,2S,590,4,4,0,0,S,N,0.00,GUAYAS,GUAYAQUIL,0.00,Fiscal,ARTE Y TECNOLOGÍA,0,3,1
5,200110286,2023,2S,590,1,0,0,1,N,AP,48.90,LOS RIOS,BABAHOYO,48.90,Particular,MATERIA INTEGRADORA DE ALIMENTOS,0,0,1
6,200118792,2023,2S,590,4,4,0,0,S,N,67.00,GUAYAS,GUAYAQUIL,67.00,Particular,ECONOMETRÍA II,0,4,0


#### Merge entre las materias vistas con la dificultad de las materias, el histórico del estdudiante y la deserción

In [80]:
df_estudiantes_resumen.shape[0], df_promedio_historico.shape[0], df_desertores.shape[0]

(8034, 7997, 368)

In [81]:
if df_estudiantes_resumen.shape[0] > df_promedio_historico.shape[0]:
    print("Hay mas estudiantes con sus materias tomadas que con su promedio historico")

Hay mas estudiantes con sus materias tomadas que con su promedio historico


In [82]:
df_materias_dificultad_historico = pd.merge(df_estudiantes_resumen, df_promedio_historico, on="COD_ESTUDIANTE", how="left") 
# deberia ser left ya que las materias con dificultad son la base principal

df_materias_dificultad_historico.shape

(8034, 26)

Aparecen estudiantes con matricula del siguiente año, y eso implica no tener promedio histórico.  
2020 2S = 202102448, 202100657, 202100368

In [83]:
df_materias_dificultad_historico[df_materias_dificultad_historico["PROMEDIO_GENERAL"].isna()]

,COD_ESTUDIANTE,ANIO,TERMINO,IDPERIODO,CANT_MATERIAS,CANT_VEZ_1,CANT_VEZ_2,CANT_VEZ_3,COMPLEMENTARIA,MATERIA_INTEGRADORA,PAIS,PROVINCIA,CANTON,PENSION,TIPOCOLEGIO,MATERIA,CANT_DIFICULTAD_ALTA,CANT_DIFICULTAD_MEDIA,CANT_DIFICULTAD_BAJA,PROMEDIO_GENERAL,MAT_TOMADAS,PROMEDIO_AP,ESNOVATO,MAT_TOMADAS_2V,MAT_TOMADAS_3V,MAT_REPROBADAS
0,,2023,2S,590,3,3,0,0,S,N,NaN,None,None,NaN,None,INGLÉS I,0,3,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7998,202370094,2023,2S,590,1,1,0,0,S,N,0.16,GUAYAS,GUAYAQUIL,0.16,Fiscal,EMPRENDIMIENTO E INNOVACIÓN,0,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7999,202370151,2023,2S,590,1,1,0,0,S,N,0.16,GUAYAS,GUAYAQUIL,0.16,Fiscal,FÚTBOL,0,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8000,202370169,2023,2S,590,1,1,0,0,S,N,0.16,GUAYAS,GUAYAQUIL,0.16,Fiscal,ANIMACIÓN I,0,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8001,202370177,2023,2S,590,1,1,0,0,S,N,0.16,GUAYAS,GUAYAQUIL,0.16,Fiscal,AJEDREZ,0,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8002,202370185,2023,2S,590,1,1,0,0,N,N,0.16,GUAYAS,GUAYAQUIL,0.16,Fiscal,DISEÑO DE EXPERIENCIA,0,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8003,202370193,2023,2S,590,1,1,0,0,N,N,0.16,GUAYAS,GUAYAQUIL,0.16,Fiscal,DISEÑO DE EXPERIENCIA,0,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8004,202370201,2023,2S,590,1,1,0,0,N,N,0.16,GUAYAS,GUAYAQUIL,0.16,Fiscal,DISEÑO DE EXPERIENCIA,0,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8005,202370219,2023,2S,590,1,1,0,0,N,N,0.16,GUAYAS,GUAYAQUIL,0.16,Fiscal,ANÁLISIS Y RESOLUCIÓN DE PROBLEMAS,0,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8006,202370227,2023,2S,590,1,1,0,0,S,N,0.16,GUAYAS,GUAYAQUIL,0.16,Fiscal,BUCEO Y ACTIVIDADES NÁUTICAS,0,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [84]:
# descartamos los nulos en promedio general
df_materias_dificultad_historico_clean = df_materias_dificultad_historico[~df_materias_dificultad_historico["PROMEDIO_GENERAL"].isna()].copy()

In [85]:
df_materias_dificultad_historico_desercion = pd.merge(df_materias_dificultad_historico_clean, df_desertores, on="COD_ESTUDIANTE", how="left") 
df_materias_dificultad_historico_desercion.shape

(7997, 28)

In [86]:
"Cantidad de estudiantes que no estan en el de desertores", df_materias_dificultad_historico_desercion["VECES"].isna().sum()

('Cantidad de estudiantes que no estan en el de desertores', np.int64(7673))

#### Creación de la variable deserción y validación con veces tomadas e integradora

In [87]:
# df_materias_dificultad_historico_desercion[df_materias_dificultad_historico_desercion["VECES"] == 0]["VECES"] = "deserción_no_registro"
# primero una copia o vista temporal con df[condicion], luego intenta asignar el valor a esa copia y no al DataFrame original
# "Chained Assignment" (asignación encadenada)

# INTEGRADORA todos son No, los que tiene 0 veces es que no se registro, desercion
df_materias_dificultad_historico_desercion.loc[df_materias_dificultad_historico_desercion["VECES"] == 0, "desercion"] = "si"
df_materias_dificultad_historico_desercion.loc[df_materias_dificultad_historico_desercion["INTEGRADORA"] != "N", "desercion"] = "no"
df_materias_dificultad_historico_desercion.loc[df_materias_dificultad_historico_desercion["VECES"].isna(), "desercion"] = "no"

In [88]:
# De respaldo
print("los que MATERIA_INTEGRADORA deberian ser desercion 'no':", df_materias_dificultad_historico_desercion[(df_materias_dificultad_historico_desercion["MATERIA_INTEGRADORA"] == "S") & (df_materias_dificultad_historico_desercion["desercion"] == "si")].shape[0])
df_materias_dificultad_historico_desercion.loc[df_materias_dificultad_historico_desercion["MATERIA_INTEGRADORA"] == "S", "desercion"] = "no"
print("despues de la actualizacion:", df_materias_dificultad_historico_desercion[(df_materias_dificultad_historico_desercion["MATERIA_INTEGRADORA"] == "S") & (df_materias_dificultad_historico_desercion["desercion"] == "si")].shape[0])

los que MATERIA_INTEGRADORA deberian ser desercion 'no': 0
despues de la actualizacion: 0


In [89]:
df_materias_dificultad_historico_desercion

,COD_ESTUDIANTE,ANIO,TERMINO,IDPERIODO,CANT_MATERIAS,CANT_VEZ_1,CANT_VEZ_2,CANT_VEZ_3,COMPLEMENTARIA,MATERIA_INTEGRADORA,PAIS,PROVINCIA,CANTON,PENSION,TIPOCOLEGIO,MATERIA,CANT_DIFICULTAD_ALTA,CANT_DIFICULTAD_MEDIA,CANT_DIFICULTAD_BAJA,PROMEDIO_GENERAL,MAT_TOMADAS,PROMEDIO_AP,ESNOVATO,MAT_TOMADAS_2V,MAT_TOMADAS_3V,MAT_REPROBADAS,INTEGRADORA,VECES,desercion
0,199201526,2023,2S,590,1,0,0,1,S,N,0.56,GUAYAS,CORONEL MARCELINO MARIDUEÑA,0.56,Particular,TURBOMAQUINARIA Y PLANTAS DE POTENCIA,0,1,0,5.25,123.0,6.87,N,35.0,12.0,66.0,NaN,NaN,no
1,199722414,2023,2S,590,5,5,0,0,S,N,0.16,GUAYAS,GUAYAQUIL,0.16,Fiscal,ANÁLISIS NUTRICIONAL DE LOS ALIMENTOS,0,5,0,7.13,49.0,7.22,N,2.0,NaN,2.0,NaN,NaN,no
2,199903519,2023,2S,590,1,1,0,0,N,AP,1.60,GUAYAS,GUAYAQUIL,1.60,Fiscal,MATERIA INTEGRADORA DE ECONOMÍA,0,1,0,6.27,97.0,7.35,N,10.0,4.0,22.0,NaN,NaN,no
3,200102978,2023,2S,590,4,4,0,0,S,N,0.00,GUAYAS,GUAYAQUIL,0.00,Fiscal,ARTE Y TECNOLOGÍA,0,3,1,6.99,32.0,7.90,N,2.0,1.0,7.0,NaN,NaN,no
4,200110286,2023,2S,590,1,0,0,1,N,AP,48.90,LOS RIOS,BABAHOYO,48.90,Particular,MATERIA INTEGRADORA DE ALIMENTOS,0,0,1,5.24,86.0,6.66,N,17.0,6.0,25.0,NaN,NaN,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7992,202319000,2023,2S,590,4,4,0,0,S,N,0.00,GUAYAS,GUAYAQUIL,0.00,Fiscal,CÁLCULO DE UNA VARIABLE,1,3,0,5.48,0.0,0.00,S,0.0,0.0,0.0,NaN,NaN,no
7993,202319018,2023,2S,590,4,4,0,0,S,N,80.00,GUAYAS,GUAYAQUIL,80.00,Particular,CÁLCULO DE UNA VARIABLE,2,2,0,5.70,0.0,0.00,S,0.0,0.0,0.0,N,0.0,si
7994,202319026,2023,2S,590,4,4,0,0,S,N,64.20,GUAYAS,GUAYAQUIL,64.20,Particular,CÁLCULO DE UNA VARIABLE,1,3,0,6.50,0.0,0.00,S,0.0,0.0,0.0,NaN,NaN,no
7995,202319034,2023,2S,590,7,7,0,0,N,N,40.00,DISTRITO CAPITAL,CARACAS,40.00,Particular,ANÁLISIS Y RESOLUCIÓN DE PROBLEMAS,0,7,0,4.10,0.0,0.00,S,0.0,0.0,0.0,NaN,NaN,no


In [90]:
cant_desercion = df_materias_dificultad_historico_desercion[df_materias_dificultad_historico_desercion["desercion"] == "si"].shape[0]
cant_veces_que_se_registra = df_materias_dificultad_historico_desercion[df_materias_dificultad_historico_desercion["VECES"] == 0].shape[0]
cant_integradora = df_materias_dificultad_historico_desercion[df_materias_dificultad_historico_desercion["INTEGRADORA"] == "N"].shape[0]
print("Cantidad de desertores:", cant_desercion)
print("Cantidad de estudiantes que no se registraron en los proximos semestres (VECES=0):", cant_veces_que_se_registra)
print("Cantidad de estudiantes que no vieron materia integradora en el futuro (INTEGRADORA=N):", cant_integradora)
# tienen que coincidir las tres cantidades, o en su defecto la suma de veces=0 e integradora=N
if cant_desercion == cant_veces_que_se_registra == cant_integradora:
    print("Las cantidades coinciden perfectamente")
elif cant_desercion == (cant_veces_que_se_registra + cant_integradora):
    print("Las cantidades coinciden en la suma de los dos casos")
else:
    print("Las cantidades no coinciden")

Cantidad de desertores: 324
Cantidad de estudiantes que no se registraron en los proximos semestres (VECES=0): 324
Cantidad de estudiantes que no vieron materia integradora en el futuro (INTEGRADORA=N): 324
Las cantidades coinciden perfectamente


In [91]:
# Eliminar columnas auxiliares
df_materias_dificultad_historico_desercion.drop(columns=["VECES", "INTEGRADORA"], inplace=True)

In [92]:
df_materias_dificultad_historico_desercion["COD_ESTUDIANTE"].nunique(), df_materias_dificultad_historico_desercion.shape[0]

(7997, 7997)

In [93]:
df_materias_dificultad_historico_desercion

,COD_ESTUDIANTE,ANIO,TERMINO,IDPERIODO,CANT_MATERIAS,CANT_VEZ_1,CANT_VEZ_2,CANT_VEZ_3,COMPLEMENTARIA,MATERIA_INTEGRADORA,PAIS,PROVINCIA,CANTON,PENSION,TIPOCOLEGIO,MATERIA,CANT_DIFICULTAD_ALTA,CANT_DIFICULTAD_MEDIA,CANT_DIFICULTAD_BAJA,PROMEDIO_GENERAL,MAT_TOMADAS,PROMEDIO_AP,ESNOVATO,MAT_TOMADAS_2V,MAT_TOMADAS_3V,MAT_REPROBADAS,desercion
0,199201526,2023,2S,590,1,0,0,1,S,N,0.56,GUAYAS,CORONEL MARCELINO MARIDUEÑA,0.56,Particular,TURBOMAQUINARIA Y PLANTAS DE POTENCIA,0,1,0,5.25,123.0,6.87,N,35.0,12.0,66.0,no
1,199722414,2023,2S,590,5,5,0,0,S,N,0.16,GUAYAS,GUAYAQUIL,0.16,Fiscal,ANÁLISIS NUTRICIONAL DE LOS ALIMENTOS,0,5,0,7.13,49.0,7.22,N,2.0,NaN,2.0,no
2,199903519,2023,2S,590,1,1,0,0,N,AP,1.60,GUAYAS,GUAYAQUIL,1.60,Fiscal,MATERIA INTEGRADORA DE ECONOMÍA,0,1,0,6.27,97.0,7.35,N,10.0,4.0,22.0,no
3,200102978,2023,2S,590,4,4,0,0,S,N,0.00,GUAYAS,GUAYAQUIL,0.00,Fiscal,ARTE Y TECNOLOGÍA,0,3,1,6.99,32.0,7.90,N,2.0,1.0,7.0,no
4,200110286,2023,2S,590,1,0,0,1,N,AP,48.90,LOS RIOS,BABAHOYO,48.90,Particular,MATERIA INTEGRADORA DE ALIMENTOS,0,0,1,5.24,86.0,6.66,N,17.0,6.0,25.0,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7992,202319000,2023,2S,590,4,4,0,0,S,N,0.00,GUAYAS,GUAYAQUIL,0.00,Fiscal,CÁLCULO DE UNA VARIABLE,1,3,0,5.48,0.0,0.00,S,0.0,0.0,0.0,no
7993,202319018,2023,2S,590,4,4,0,0,S,N,80.00,GUAYAS,GUAYAQUIL,80.00,Particular,CÁLCULO DE UNA VARIABLE,2,2,0,5.70,0.0,0.00,S,0.0,0.0,0.0,si
7994,202319026,2023,2S,590,4,4,0,0,S,N,64.20,GUAYAS,GUAYAQUIL,64.20,Particular,CÁLCULO DE UNA VARIABLE,1,3,0,6.50,0.0,0.00,S,0.0,0.0,0.0,no
7995,202319034,2023,2S,590,7,7,0,0,N,N,40.00,DISTRITO CAPITAL,CARACAS,40.00,Particular,ANÁLISIS Y RESOLUCIÓN DE PROBLEMAS,0,7,0,4.10,0.0,0.00,S,0.0,0.0,0.0,no


In [94]:
# Rellenar solo las columnas que existen
columnas_a_rellenar = ["MAT_TOMADAS_2V", "MAT_TOMADAS_3V", "MAT_REPROBADAS"]
for col in columnas_a_rellenar:
    if col in df_materias_dificultad_historico_desercion.columns:
        print("Procesar:", col)
        df_materias_dificultad_historico_desercion[col] = df_materias_dificultad_historico_desercion[col].fillna(0.0)
        df_materias_dificultad_historico_desercion[col] = df_materias_dificultad_historico_desercion[col].astype(float)

Procesar: MAT_TOMADAS_2V
Procesar: MAT_TOMADAS_3V
Procesar: MAT_REPROBADAS


In [95]:
df_materias_dificultad_historico_desercion.describe()

,ANIO,IDPERIODO,CANT_MATERIAS,CANT_VEZ_1,CANT_VEZ_2,CANT_VEZ_3,PAIS,PENSION,CANT_DIFICULTAD_ALTA,CANT_DIFICULTAD_MEDIA,CANT_DIFICULTAD_BAJA,PROMEDIO_GENERAL,MAT_TOMADAS,PROMEDIO_AP,MAT_TOMADAS_2V,MAT_TOMADAS_3V,MAT_REPROBADAS
count,7997.0,7997.0,7997.000000,7997.000000,7997.000000,7997.000000,7997.00000,7997.00000,7997.000000,7997.000000,7997.000000,7997.000000,7997.000000,7997.000000,7997.000000,7997.000000,7997.000000
mean,2023.0,590.0,4.119045,3.734776,0.310992,0.073277,129.44678,129.44678,0.274728,3.310992,0.533325,5.989537,28.024884,5.344348,1.890209,0.286607,3.396399
std,0.0,0.0,1.479756,1.747880,0.639062,0.299879,135.72395,135.72395,0.601339,1.567378,0.763110,1.277813,19.818903,2.286350,2.892427,0.764316,5.034267
min,2023.0,590.0,1.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2023.0,590.0,3.000000,3.000000,0.000000,0.000000,0.00000,0.00000,0.000000,2.000000,0.000000,5.320000,12.000000,4.750000,0.000000,0.000000,0.000000
50%,2023.0,590.0,4.000000,4.000000,0.000000,0.000000,101.93000,101.93000,0.000000,3.000000,0.000000,6.200000,26.000000,6.210000,1.000000,0.000000,1.000000
75%,2023.0,590.0,5.000000,5.000000,0.000000,0.000000,196.95000,196.95000,0.000000,4.000000,1.000000,6.880000,43.000000,6.860000,3.000000,0.000000,4.000000
max,2023.0,590.0,12.000000,12.000000,5.000000,3.000000,748.61000,748.61000,3.000000,10.000000,6.000000,10.000000,126.000000,9.100000,35.000000,12.000000,66.000000


In [96]:
df_materias_dificultad_historico_desercion.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7997 entries, 0 to 7996
Data columns (total 27 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   COD_ESTUDIANTE         7997 non-null   object 
 1   ANIO                   7997 non-null   int64  
 2   TERMINO                7997 non-null   object 
 3   IDPERIODO              7997 non-null   int64  
 4   CANT_MATERIAS          7997 non-null   int64  
 5   CANT_VEZ_1             7997 non-null   int64  
 6   CANT_VEZ_2             7997 non-null   int64  
 7   CANT_VEZ_3             7997 non-null   int64  
 8   COMPLEMENTARIA         7997 non-null   object 
 9   MATERIA_INTEGRADORA    7997 non-null   object 
 10  PAIS                   7997 non-null   float64
 11  PROVINCIA              7997 non-null   object 
 12  CANTON                 7997 non-null   object 
 13  PENSION                7997 non-null   float64
 14  TIPOCOLEGIO            7997 non-null   object 
 15  MATE

In [97]:
# validacion que ninguna columna tenga nulos
nulos_por_columna = df_materias_dificultad_historico_desercion.isna().sum()
nulos_por_columna[nulos_por_columna > 0]
# nulos_por_columna

Series([], dtype: int64)

#### Merge con socioeconómico

In [98]:
df_socioeconomico["CODESTUDIANTE"].nunique(), df_socioeconomico.shape[0], df_materias_dificultad_historico_desercion.shape[0]

(17841, 17841, 7997)

In [99]:
df_materias_dificultad_historico_desercion_new = df_materias_dificultad_historico_desercion.copy()

In [100]:
# # df_materias_dificultad_historico_desercion = COD_ESTUDIANTE
# # df_socioeconomico = CODESTUDIANTE
# df_materias_dificultad_historico_desercion_new = pd.merge(
#     df_materias_dificultad_historico_desercion,
#     df_socioeconomico,
#     left_on="COD_ESTUDIANTE",
#     right_on="CODESTUDIANTE",
#     how="left"
# )

# df_materias_dificultad_historico_desercion_new.shape

In [101]:
if "ESTACOMPLETA" in df_materias_dificultad_historico_desercion_new.columns:
    print("Cantidad de registros sin datos socioeconomicos", df_materias_dificultad_historico_desercion_new["ESTACOMPLETA"].isna().sum())
else:
    print("Cantidad de registros", df_materias_dificultad_historico_desercion_new.shape[0] - df_materias_dificultad_historico_desercion_new["desercion"].isna().sum())

Cantidad de registros 7997


Cantidad de descartados por falta de factor socioeconomico del 13/01/26:    
2020 1S 1234  
2020 2S 1159  
2021 1S 1033  
2021 2S 929  
2022 1S 113  
2022 2S 81  
2023 1S 50  
2023 2S 16  


Todos los datos sin factor socioeconomico 15/01/26:    
2020 1S 9466  
2020 2S 9105  
2021 1S 8753  
2021 2S 8368  
2022 1S 8190  
2022 2S 8039  
2023 1S 8087  
2023 2S 7997  

In [102]:
df_materias_dificultad_historico_desercion_new

,COD_ESTUDIANTE,ANIO,TERMINO,IDPERIODO,CANT_MATERIAS,CANT_VEZ_1,CANT_VEZ_2,CANT_VEZ_3,COMPLEMENTARIA,MATERIA_INTEGRADORA,PAIS,PROVINCIA,CANTON,PENSION,TIPOCOLEGIO,MATERIA,CANT_DIFICULTAD_ALTA,CANT_DIFICULTAD_MEDIA,CANT_DIFICULTAD_BAJA,PROMEDIO_GENERAL,MAT_TOMADAS,PROMEDIO_AP,ESNOVATO,MAT_TOMADAS_2V,MAT_TOMADAS_3V,MAT_REPROBADAS,desercion
0,199201526,2023,2S,590,1,0,0,1,S,N,0.56,GUAYAS,CORONEL MARCELINO MARIDUEÑA,0.56,Particular,TURBOMAQUINARIA Y PLANTAS DE POTENCIA,0,1,0,5.25,123.0,6.87,N,35.0,12.0,66.0,no
1,199722414,2023,2S,590,5,5,0,0,S,N,0.16,GUAYAS,GUAYAQUIL,0.16,Fiscal,ANÁLISIS NUTRICIONAL DE LOS ALIMENTOS,0,5,0,7.13,49.0,7.22,N,2.0,0.0,2.0,no
2,199903519,2023,2S,590,1,1,0,0,N,AP,1.60,GUAYAS,GUAYAQUIL,1.60,Fiscal,MATERIA INTEGRADORA DE ECONOMÍA,0,1,0,6.27,97.0,7.35,N,10.0,4.0,22.0,no
3,200102978,2023,2S,590,4,4,0,0,S,N,0.00,GUAYAS,GUAYAQUIL,0.00,Fiscal,ARTE Y TECNOLOGÍA,0,3,1,6.99,32.0,7.90,N,2.0,1.0,7.0,no
4,200110286,2023,2S,590,1,0,0,1,N,AP,48.90,LOS RIOS,BABAHOYO,48.90,Particular,MATERIA INTEGRADORA DE ALIMENTOS,0,0,1,5.24,86.0,6.66,N,17.0,6.0,25.0,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7992,202319000,2023,2S,590,4,4,0,0,S,N,0.00,GUAYAS,GUAYAQUIL,0.00,Fiscal,CÁLCULO DE UNA VARIABLE,1,3,0,5.48,0.0,0.00,S,0.0,0.0,0.0,no
7993,202319018,2023,2S,590,4,4,0,0,S,N,80.00,GUAYAS,GUAYAQUIL,80.00,Particular,CÁLCULO DE UNA VARIABLE,2,2,0,5.70,0.0,0.00,S,0.0,0.0,0.0,si
7994,202319026,2023,2S,590,4,4,0,0,S,N,64.20,GUAYAS,GUAYAQUIL,64.20,Particular,CÁLCULO DE UNA VARIABLE,1,3,0,6.50,0.0,0.00,S,0.0,0.0,0.0,no
7995,202319034,2023,2S,590,7,7,0,0,N,N,40.00,DISTRITO CAPITAL,CARACAS,40.00,Particular,ANÁLISIS Y RESOLUCIÓN DE PROBLEMAS,0,7,0,4.10,0.0,0.00,S,0.0,0.0,0.0,no


#### Clean columns

In [103]:
if "NUMEROSFAMILIARES" in df_materias_dificultad_historico_desercion_new.columns:
    df_materias_dificultad_historico_desercion_new["NUMEROSFAMILIARES"] = df_materias_dificultad_historico_desercion_new["NUMEROSFAMILIARES"].fillna(0).astype(int)

In [104]:
# Asegurar tipos de datos consistentes para evitar problemas de tipos mixtos
columnas_a_string = [
    'TERMINO', 'MATERIA_INTEGRADORA',
    'FECHACREACION', 'FECHAENVIOUBEP', 'FECHAENVIO', 'APELLIDOS', 'NOMBRES', 'EMAIL',
    'P1_NACIONALIDAD', 'ANIO_TERMINO_INGRESO', 'TIENEDISCAPACIDAD', 'TIPODISCAPACIDAD',
    'SEXO', 'AUTOIDENTIFICACIONGENERO', 'AUTOIDENTIFICACIONETNICA', 'ESTADOCIVIL',
    'FECHANACIMIENTO', 'PAISNACIMIENTO', 'PROVINCIANACIMIENTO', 'CIUDADNACIMIENTO',
    'TELEFONOCELULAR', 'TELEFONOFIJO', 'CORREOALTERNO', 'COLEGIO', 'PAISCOLEGIO',
    'PROVINCIACOLEGIO', 'CANTONCOLEGIO', 'TIPOCOLEGIO', 'JORNADA', 'BECACOLEGIO',
    'OTROSIDIOMAS', 'IDIOMAS', 'COBOCIMIENTOINGLESPOR', 'FRASETRESCOMIDAS',
    'FRASEHABITOALIMENTICIO', 'FRASEHABITOALIMENTICIOV2', 'TIEMPOPROMEDIOLLEGARESPOL',
    'TIEMPOPROMEDIOBICICLETAENTRADAESPOL', 'TIEMPOPROMEDIOCAMINATAENTRADAESPOL',
    'TIEMPOPROMEDIOBICICLETASALIDAESPOL', 'TIEMPOPROMEDIOCAMINATASALIDAESPOL',
    'NIVELINGLES', 'POSEETARJETACREDITO', 'POSEETARJETADEBITO', 'CUENTASBANCO',
    'HERMANOSESTUDIANDOESPOL', 'NIVELINSTRUCCIONPADRE', 'NIVELINSTRUCCIONMADRE',
    'ESTADOCIVILPADRES', 'DISCAPACIDAD', 'FAMILIARDISCAPACIDAD', 'ENFERMEDAD',
    'FAMILIARENFERMEDAD', 'RECIBEBONO', 'DIFICULTADAPRENDIZAJE', 'PAISVIVE',
    'PROVINCIAVIVE', 'CIUDADVIVE', 'PARROQUIAVIVE', 'DIRECCION', 'COORDENADAS',
    'TIPOPARROQUIA', 'VIVEGRUPOFAMILIAR', 'PAISVIVESEP', 'PROVINCIAVIVIENDASEP',
    'CANTONVIVESEP', 'PARROQUIAVIVESEP', 'DIRECCIONVIVSEP', 'TIPOVIVIENDASEP',
    'ESTADOVIVIENDASEP', 'SALA', 'COMEDOR', 'ESTUDIO', 'COCINA', 'LAVANDERIA',
    'GARAJE', 'METERIALTECHOVIVIENDASEP', 'METERIALPISOVIVIENDASEP',
    'METERIALPAREDVIVIENDASEP', 'VIAACCESOVIVIENDASEP', 'ABASTECIMIENTOAGUA',
    'SERVHIGIENE', 'ELIMINACIONBASURA', 'SERVICIOELECTRICIDAD', 'POSEEVEHICULO',
    'RECIBEAYUDA', 'PARIENTEAYUDA', 'TIPOBACHILLER', 'COMIDAS', 'DISPOSITIVOS',
    'MANEJOCELULAR', 'ACCESOINTERNET', 'COMPLEMENTARIA', 'PROVINCIA', 'CANTON',
    'TIPOCOLEGIO_x', 'MATERIA', 'ESNOVATO', 'TIPOCOLEGIO_y'
]

# Convertir columnas a string, manejando NaN primero
for col in columnas_a_string:
    if col in df_materias_dificultad_historico_desercion_new.columns:
        df_materias_dificultad_historico_desercion_new[col] = df_materias_dificultad_historico_desercion_new[col].fillna('').astype(str).str.strip().str.lower()


# NUMEROIDENTIFICACION, ANIO_TERMINO_INGRESO, TELEFONOCELULAR, TELEFONOFIJO mantenerla como string también para evitar problemas con cédulas
for col in ['NUMEROIDENTIFICACION', 'ANIO_TERMINO_INGRESO', 'TELEFONOCELULAR', 'TELEFONOFIJO']:
    if col in df_materias_dificultad_historico_desercion_new.columns:
        df_materias_dificultad_historico_desercion_new[col] = df_materias_dificultad_historico_desercion_new[col].fillna('').astype(str).str.strip()
print(f"Tipos de datos asegurados para {len([c for c in columnas_a_string if c in df_materias_dificultad_historico_desercion_new.columns])} columnas")


Tipos de datos asegurados para 8 columnas


In [105]:
df_materias_dificultad_historico_desercion_new.to_csv(f"../data/desercion/merge/materias_dificultad_historico_desercion_{anio_term}.csv", index=False)